# 06 · Contact patterns and cardiovascular hospitalization: main analyses

Builds the analysis frame for the landmark population, imputes missing covariates, and fits the main association models.

**Reads (`DATA_DIR`):** `landmark_build` (02), `covariates` (01), `trajectory_classes` (05)

**Produces (patient-level, `DATA_DIR`):** `covariates_preindex`, `covariates_oldwindow`, `analysis_frame`, `mi` (30 imputations)

**Produces (aggregate, `results/`):** `06_main.json` (every pooled estimate), `table4.csv`, `tableS7.csv`, `figureS2_gap_spline.csv`, `gap_cutoffs.csv`, `uno_c.csv`, `figure3b.csv`, `figure3cd.csv`, `spline_knots.json`

**Paper items:** Methods, *Outcome and covariates* and *Association analyses*; Table 4; Table S7; Figure 3B–D (3A data come from notebook 05); Figure S2; proportional-hazards check, balance after weighting, imputation diagnostics, and minimum detectable hazard ratios in the text. Sensitivity analyses and outcome components are in notebook 06b.

**Runtime:** roughly 1.5–3 hours (30 imputations × several models, Fine–Gray in R, Uno's C bootstrap). Results are saved section by section.

## 0 · Setup

Patients are sorted by ID before imputation, and every random step is seeded, so results are repeatable.

In [ ]:
# ---- edit these ----
REPO_DIR   = "/content/drive/MyDrive/Ronveer Sandhu/Project 1 -MIMIC analysis-hypertension retention/htn-bp-followup-mimic"
MIMIC_HOSP = "/content/drive/MyDrive/MIMIC Data/mimic-iv-2.2/hosp"
MIMIC_ED   = "/content/drive/MyDrive/MIMIC Data/mimic-iv-2.2/ed"
DATA_DIR   = "/content/drive/MyDrive/htn_bp_data"

M_IMPUTATIONS = 30

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!pip -q install duckdb==1.3.2 lifelines==0.30.3 scikit-learn==1.6.1 statsmodels==0.15.0

import os, sys, json, time, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, os.path.join(REPO_DIR, "src"))
from htn_utils import *
from landmark_models import *

import numpy as np
import pandas as pd
from lifelines import CoxPHFitter, NelsonAalenFitter, KaplanMeierFitter, AalenJohansenFitter

RESULTS_DIR = os.path.join(REPO_DIR, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

lb  = load_df(DATA_DIR, "landmark_build")
cov = load_df(DATA_DIR, "covariates")
cls = load_df(DATA_DIR, "trajectory_classes")
TRAJ = json.load(open(os.path.join(RESULTS_DIR, "05_trajectories.json")))
set_class_order(TRAJ["class_order"]); KEY = key_class()
print(f"{TRAJ['k_selected']} trajectory classes:", ORDER)
lb["index_date"] = pd.to_datetime(lb.index_date)

## 1 · Covariates measured before index

Blood pressure, labs, and body mass index use the **closest value in the 365 days up to and including index** (same-day values averaged); the antihypertensive drug count covers the 365 days up to and including index. A window average from 365 days before to 90 days after index is also built for the Table S3 sensitivity analysis. Built for everyone in any landmark sample (primary, day-730-information, events-before-index kept).

In [ ]:
con = connect(MIMIC_HOSP, MIMIC_ED)
need = lb[lb.in_landmark | lb.in_day730_info | lb.in_landmark_prior_ok]
pt = need[["subject_id", "index_date"]].copy(); pt["index_date"] = pt.index_date.dt.date
con.register("pt", pt)
t0 = time.time()

bp = con.execute('''
  SELECT o.subject_id, CAST(o.chartdate AS DATE) - pt.index_date AS day,
         TRY_CAST(SPLIT_PART(o.result_value, '/', 1) AS DOUBLE) AS sbp,
         TRY_CAST(SPLIT_PART(o.result_value, '/', 2) AS DOUBLE) AS dbp
  FROM omr o JOIN pt ON pt.subject_id = o.subject_id
  WHERE o.result_name LIKE 'Blood Pressure%'
    AND CAST(o.chartdate AS DATE) BETWEEN pt.index_date - 365 AND pt.index_date + 90''').df()
bp = bp[bp.sbp.between(60, 300)]
bmi = con.execute('''
  SELECT o.subject_id, CAST(o.chartdate AS DATE) - pt.index_date AS day, TRY_CAST(o.result_value AS DOUBLE) AS bmi
  FROM omr o JOIN pt ON pt.subject_id = o.subject_id
  WHERE o.result_name LIKE 'BMI%'
    AND CAST(o.chartdate AS DATE) BETWEEN pt.index_date - 365 AND pt.index_date + 90''').df()
bmi = bmi[bmi.bmi.between(12, 80)]
labs = con.execute('''
  SELECT l.subject_id, CAST(l.charttime AS DATE) - pt.index_date AS day, l.itemid, l.valuenum
  FROM labevents l JOIN pt ON pt.subject_id = l.subject_id
  WHERE l.itemid IN (50912, 50852) AND l.valuenum IS NOT NULL
    AND CAST(l.charttime AS DATE) BETWEEN pt.index_date - 365 AND pt.index_date + 90''').df()
cr  = labs[labs.itemid == 50912].rename(columns={"valuenum": "creatinine"})
a1c = labs[labs.itemid == 50852].rename(columns={"valuenum": "hba1c"})
print(f"rows: BP {len(bp):,} | BMI {len(bmi):,} | creatinine {len(cr):,} | HbA1c {len(a1c):,}  [{time.time()-t0:.0f}s]")

def window_mean(d, cols):                 # sensitivity window: average over -365..+90
    return d.groupby("subject_id")[cols].mean()
def closest_pre(d, cols):                 # latest date in -365..0, same-day values averaged
    d = d[d.day <= 0]
    last = d.groupby("subject_id").day.transform("max")
    return d[d.day == last].groupby("subject_id")[cols].mean()
def egfr_2021(scr, age, female):          # CKD-EPI 2021, no race term
    k = np.where(female, 0.7, 0.9); a = np.where(female, -0.241, -0.302); r = scr / k
    return 142 * np.minimum(r, 1) ** a * np.maximum(r, 1) ** -1.2 * 0.9938 ** age * np.where(female, 1.012, 1)

base = need.set_index("subject_id")[["age_index"]].join(cov.set_index("subject_id")[["gender"]])
def assemble(fn):
    out = (base.join(fn(bp, ["sbp", "dbp"])).join(fn(bmi, ["bmi"]))
               .join(fn(cr, ["creatinine"])).join(fn(a1c, ["hba1c"])))
    out["egfr"] = egfr_2021(out.creatinine, out.age_index, out.gender.eq("F"))
    return out.rename(columns={"sbp": "sbp_baseline", "dbp": "dbp_baseline"}).drop(columns=["age_index", "gender"])

pre = assemble(closest_pre).reset_index()
old_window = assemble(window_mean).reset_index()

drugs = con.execute(f'''
  SELECT pt.subject_id, COUNT(DISTINCT LOWER(r.drug)) AS n_antihtn_drugs
  FROM pt JOIN prescriptions r ON r.subject_id = pt.subject_id
  WHERE {sql_contains('r.drug', ANTIHTN)}
    AND CAST(r.starttime AS DATE) BETWEEN pt.index_date - 365 AND pt.index_date
  GROUP BY pt.subject_id''').df()
pre = pre.merge(drugs, on="subject_id", how="left").fillna({"n_antihtn_drugs": 0})

## 2 · Analysis frame

Primary landmark population with pre-index covariates, trajectory class (with posterior probabilities), the longest gap (continuous and > 365 days), and the follow-up from notebook 02. Prior-use counts enter on the log scale.

In [ ]:
def build_frame(rows):
    keep_cov = ["subject_id", "gender", "race", "insurance", "n_ed_prior", "n_inp_prior", "n_enc_prior",
                "elixhauser_vw", "chf", "rf"]
    F = (rows.merge(cov[keep_cov], on="subject_id", how="left")
             .merge(pre, on="subject_id", how="left"))
    F["ins_grp"] = np.where(F.insurance.eq("Medicaid"), "Medicaid", np.where(F.insurance.eq("Medicare"), "Medicare", "Other"))
    for s, c in [("log_enc_prior", "n_enc_prior"), ("log_ed_prior", "n_ed_prior"),
                 ("log_inp_prior", "n_inp_prior"), ("log_idx_los", "idx_los")]:
        F[s] = np.log1p(F[c].clip(lower=0))
    F["gap_days"] = F.longest_gap.astype(float)
    F["gap365"]   = F.gapped.astype(int)
    return F

A = build_frame(lb[lb.in_landmark]).merge(
        cls[["subject_id", "label", "max_post"] + [c for c in cls.columns if c.startswith("p_")]], on="subject_id")

A = A.sort_values("subject_id").reset_index(drop=True)
print(f"landmark population: {len(A):,}")
print("events:", int((A.status == 1).sum()), "| deaths:", int((A.status == 2).sum()))
print("classes:", A.label.value_counts().reindex(ORDER).to_dict())
print("missing (%):", (100 * A[MISS].isna().mean()).round(1).to_dict())

SPL = ["age_index", "sbp_baseline", "egfr", "bmi", "elixhauser_vw", "log_enc_prior",
       "log_ed_prior", "log_inp_prior", "log_idx_los", "n_antihtn_drugs"]
KN = make_knots(A, SPL)
print("\nknots:", {k: v for k, v in KN.items()})

T  = A.time.astype(float).values
E1 = (A.status == 1).astype(int).values
E2 = (A.status == 2).astype(int).values
S  = A.status.astype(int).values
Cl = class_dummies(A); G1 = A[["gap365"]].astype(float); GS = gap_spline(A, KN)

## 3 · Multiple imputation (m = 30)

Chained equations (Bayesian ridge, posterior draws) for eGFR, body mass index, and glycated hemoglobin. The imputation model includes all covariates, both contact measures, and, for the event and the competing death, the event indicator and the Nelson–Aalen cumulative hazard. Imputed values are limited to the 0.5th–99.5th percentiles of observed values.

In [ ]:
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge

H1 = NelsonAalenFitter().fit(T, E1).cumulative_hazard_at_times(T).values
H2 = NelsonAalenFitter().fit(T, E2).cumulative_hazard_at_times(T).values
Xraw = build_X(A, KN, linear=True)
imp_in = pd.concat([Xraw, class_dummies(A), A[["gap_days"]],
                    pd.DataFrame({"E1": E1, "E2": E2, "H1": H1, "H2": H2})], axis=1)
mask = {c: A[c].isna().values for c in MISS}
lims = {c: (np.nanpercentile(A[c], 0.5), np.nanpercentile(A[c], 99.5)) for c in MISS}

M = M_IMPUTATIONS; t0 = time.time(); rows = []
for m in range(M):
    imp = IterativeImputer(estimator=BayesianRidge(), sample_posterior=True, max_iter=10, random_state=m)
    Z = pd.DataFrame(imp.fit_transform(imp_in), columns=imp_in.columns)
    d = A[["subject_id"]].copy(); d[".imp"] = m + 1
    for c in MISS:
        d[c] = np.where(mask[c], Z[c].clip(*lims[c]), A[c])
    rows.append(d)
    if (m + 1) % 10 == 0: print(f"  imputed {m + 1}/{M} [{(time.time() - t0) / 60:.1f} min]")
MI = pd.concat(rows, ignore_index=True)

imp_means = {}
for c in MISS:
    imp_means[c] = (float(MI.loc[np.tile(mask[c], M), c].mean()), float(A[c].mean()))
    print(f"  {c:6s} imputed mean {imp_means[c][0]:.2f} | observed mean {imp_means[c][1]:.2f}")

def imputed(m):
    F = A.copy(); F[MISS] = MI.loc[MI[".imp"] == m, MISS].values; return F

## 4 · Cause-specific Cox models (Table 4, primary)

In [ ]:
def fit_cs(F, expo, idx=None, T_=T, E_=E1, extra=None, linear=False):
    parts = [pd.DataFrame({"T": T_, "E": E_}), build_X(F, KN, linear=linear)] + ([extra] if extra is not None else []) + [expo]
    d = pd.concat(parts, axis=1)
    if idx is not None: d = d.loc[idx]
    d = d.loc[:, d.std() > 0]
    c = CoxPHFitter(penalizer=1e-6).fit(d, "T", "E")
    k = [x for x in expo.columns if x in d.columns]
    return c.params_[k].values, c.variance_matrix_.loc[k, k].values, k

EXPOS = {"class": Cl, "gap365": G1, "gap_spline": GS, "class+gap365": pd.concat([Cl, G1], axis=1)}
res = {k: [] for k in list(EXPOS) + ["gap365_in_Minimal"]}
minimal = A.index[A.label == "Minimal Contact"]
t0 = time.time()
for m in range(1, M + 1):
    F = imputed(m)
    for k, ex in EXPOS.items(): res[k].append(fit_cs(F, ex))
    res["gap365_in_Minimal"].append(fit_cs(F, G1, idx=minimal))
    if m % 10 == 0: print(f"  {m}/{M} imputations [{(time.time()-t0)/60:.1f} min]")

OUTR = {}
for k in res:
    tab, w, p = pool(res[k])
    OUTR[k] = {"table": tab.to_dict("index"), "wald": w, "df": len(tab), "p_overall": p}
    print(f"=== {k} (overall p = {p:.3f}) ===\n{tab.round(3).to_string()}\n")
_, _, OUTR["class_given_gap_p"] = pool_subset(res["class+gap365"], list(Cl.columns))
_, _, OUTR["gap_given_class_p"] = pool_subset(res["class+gap365"], ["gap365"])
print(f"class given gap p = {OUTR['class_given_gap_p']:.2f} | gap given class p = {OUTR['gap_given_class_p']:.2f}")
OUTR["max_FMI"] = float(max(pd.DataFrame(OUTR[k]["table"]).T.FMI.max() for k in res))
print(f"largest fraction of missing information: {OUTR['max_FMI']:.3f}")

## 5 · Fine–Gray and doubly robust models

- **Fine–Gray**: subdistribution hazards with death as the competing event (R `survival::finegray`).
- **Doubly robust**: stabilized inverse probability weights from a multinomial (class) or logistic (gap) model of exposure on covariates, truncated at the 1st and 99th percentiles, plus covariate adjustment; robust standard errors.

In [ ]:
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
ro.r("suppressMessages(library(survival))")

def fit_fg(F, expo):
    d = pd.concat([pd.DataFrame({"T": T, "S": S}), build_X(F, KN), expo], axis=1)
    d = d[["T", "S"] + [c for c in d.columns[2:] if d[c].std() > 0]]
    nm = {c: f"v{i}" for i, c in enumerate(d.columns) if c not in ("T", "S")}
    d = d.rename(columns=nm); ek = [nm[c] for c in expo.columns if c in nm]
    with localconverter(ro.default_converter + pandas2ri.converter):
        ro.globalenv["d"] = d
    ro.globalenv["ek"] = ro.StrVector(ek)
    ro.r('''
      d$S <- factor(d$S, levels = c(0, 1, 2))
      fg  <- finegray(Surv(T, S) ~ ., data = d, etype = "1")
      vv  <- setdiff(names(d), c("T", "S"))
      fit <- coxph(as.formula(paste("Surv(fgstart, fgstop, fgstatus) ~", paste(vv, collapse = "+"))),
                   data = fg, weights = fgwt)
      b <- coef(fit)[ek]; V <- vcov(fit)[ek, ek, drop = FALSE]''')
    return np.array(ro.r("b")), np.array(ro.r("V")), [c for c in expo.columns if c in nm]

def iptw(F, a):
    X = build_X(F, KN); Z = StandardScaler().fit_transform(X)
    ps = LogisticRegression(max_iter=5000).fit(Z, a); P = ps.predict_proba(Z)
    cl_ = list(ps.classes_); pa = P[np.arange(len(a)), [cl_.index(x) for x in a]]
    marg = pd.Series(a).map(pd.Series(a).value_counts(normalize=True)).values
    return X, np.clip(marg / pa, *np.percentile(marg / pa, [1, 99]))

def fit_dr(F, expo, a):
    X, w = iptw(F, a)
    d = pd.concat([pd.DataFrame({"T": T, "E": E1, "w": w}), X, expo], axis=1)
    d = d.loc[:, (d.std() > 0) | d.columns.isin(["T", "E", "w"])]
    c = CoxPHFitter(penalizer=1e-6).fit(d, "T", "E", weights_col="w", robust=True)
    k = [x for x in expo.columns if x in d.columns]
    return c.params_[k].values, c.variance_matrix_.loc[k, k].values, k

a_cls, a_gap = A.label.values, A.gap365.values
RES = {"fg_class": [], "fg_gap365": [], "dr_class": [], "dr_gap365": []}
t0 = time.time()
for m in range(1, M + 1):
    F = imputed(m)
    RES["fg_class"].append(fit_fg(F, Cl)); RES["fg_gap365"].append(fit_fg(F, G1))
    RES["dr_class"].append(fit_dr(F, Cl, a_cls)); RES["dr_gap365"].append(fit_dr(F, G1, a_gap))
    if m % 5 == 0: print(f"  {m}/{M} [{(time.time()-t0)/60:.1f} min]")
for k in RES:
    tab, w, p = pool(RES[k]); OUTR[k] = {"table": tab.to_dict("index"), "wald": w, "p_overall": p}
    print(f"=== {k} (overall p = {p:.3f}) ===\n{tab.round(3).to_string()}\n")

In [ ]:
# balance after weighting (imputation 1): largest standardized difference per comparison
def smd_table(X, w, a, ref):
    rows = []
    for g in [x for x in np.unique(a) if x != ref]:
        i, j = a == g, a == ref
        v = np.sqrt((X[i].var(0) + X[j].var(0)) / 2).replace(0, 1)
        after = (pd.Series(np.average(X[i], 0, w[i]), X.columns) - pd.Series(np.average(X[j], 0, w[j]), X.columns)).abs() / v
        rows.append(pd.DataFrame({"comparison": f"{g} vs {ref}", "covariate": X.columns, "after": after.values}))
    return pd.concat(rows)

F1 = imputed(1); BAL = {}
for name, a, ref in [("class", a_cls, "Minimal Contact"), ("gap", a_gap, 0)]:
    X1, w1 = iptw(F1, a)
    t = smd_table(X1, w1, a, ref)
    top = t.sort_values("after", ascending=False).iloc[0]
    BAL[name] = t.groupby("comparison").after.max().round(3).to_dict()
    print(f"{name}: largest SMD after weighting by comparison {BAL[name]} | worst: {top.covariate} ({top.after:.2f})")
OUTR["balance_max_smd"] = BAL

## 6 · Gap dose-response (Figure S2) and secondary cutoffs

Restricted cubic spline of the longest interval without contact (knots at the 5th/35th/65th/95th percentiles), hazard ratio relative to a 90-day interval; and gaps longer than 180, 270, and 365 days.

In [ ]:
qb, Tt = pool_full(res["gap_spline"])
grid = np.arange(14, 731, 2); REF = 90
C = rcs(grid, KN["gap_days"]) - rcs(np.array([REF]), KN["gap_days"])[0]
lhr = C @ qb; se = np.sqrt(np.einsum("ij,jk,ik->i", C, Tt, C))
curve = pd.DataFrame({"gap_days": grid, "HR": np.exp(lhr), "lo": np.exp(lhr - 1.96 * se), "hi": np.exp(lhr + 1.96 * se)})
print(f"spline overall p = {OUTR['gap_spline']['p_overall']:.2f}")
print(curve[curve.gap_days.isin([180, 270, 366, 730])].round(2).to_string(index=False))

cut_rows = []
for cut in [180, 270, 365]:
    Gc = pd.DataFrame({f"gap_gt{cut}": (A.gap_days > cut).astype(float)})
    tab = pool([fit_cs(imputed(m), Gc) for m in range(1, M + 1)])[0]
    cut_rows.append(tab.assign(prevalence_pct=100 * (A.gap_days > cut).mean()))
cuts = pd.concat(cut_rows)
print(cuts.round(3).to_string())
OUTR["gap_cutoffs"] = cuts.to_dict("index")

## 7 · Added information: Uno's C-statistic at 730 days (Table 4B)

Uno's C with competing deaths treated as censoring, averaged over imputations; confidence intervals for the gain from 1,000 bootstrap samples of imputation 1 with fixed linear predictors. Apparent, not corrected for optimism.

In [ ]:
TAU = 730
G_at = KaplanMeierFitter().fit(T, 1 - E1).survival_function_at_times(T - 1e-9).values

def uno_c(eta, idx=None):
    t, e, g, h = (T, E1, G_at, eta) if idx is None else (T[idx], E1[idx], G_at[idx], eta[idx])
    num = den = 0.0
    for i in np.where((e == 1) & (t < TAU))[0]:
        risk = t > t[i]; w = 1.0 / g[i] ** 2
        den += w * risk.sum()
        num += w * ((h[risk] < h[i]).sum() + 0.5 * (h[risk] == h[i]).sum())
    return num / den

def lp(F, expo=None):
    d = pd.concat([pd.DataFrame({"T": T, "E": E1}), build_X(F, KN)] + ([expo] if expo is not None else []), axis=1)
    d = d.loc[:, d.std() > 0]
    return CoxPHFitter(penalizer=1e-6).fit(d, "T", "E").predict_log_partial_hazard(d).values

SETS = [("covariates", None), ("+ gap", G1), ("+ class", Cl), ("+ gap + class", pd.concat([Cl, G1], axis=1))]
t0 = time.time()
Cs = {k: [uno_c(lp(imputed(m), ex)) for m in range(1, M + 1)] for k, ex in SETS}
Cbar = {k: float(np.mean(v)) for k, v in Cs.items()}
F1 = imputed(1); LP = {k: lp(F1, ex) for k, ex in SETS}
rng = np.random.default_rng(42); D = {k: [] for k, _ in SETS[1:]}
for r in range(1000):
    b = rng.integers(0, len(T), len(T)); c0 = uno_c(LP["covariates"], b)
    for k in D: D[k].append(uno_c(LP[k], b) - c0)
    if (r + 1) % 250 == 0: print(f"  bootstrap {r+1}/1000 [{(time.time()-t0)/60:.1f} min]")
uno = pd.DataFrame([{"model": "covariates", "C": Cbar["covariates"], "gain": np.nan, "lo": np.nan, "hi": np.nan}] +
                   [{"model": k, "C": Cbar[k], "gain": Cbar[k] - Cbar["covariates"],
                     "lo": np.percentile(D[k], 2.5), "hi": np.percentile(D[k], 97.5)} for k in D])
print(uno.round(4).to_string(index=False))
OUTR["uno_c"] = uno.to_dict("records")

## 8 · Cumulative incidence, Gray's tests, Table S7, and Figure 3B–D data

In [ ]:
ro.r('if (!requireNamespace("cmprsk", quietly = TRUE)) install.packages("cmprsk", repos = "https://cloud.r-project.org", quiet = TRUE)')
ro.r("suppressMessages(library(cmprsk))")
def gray_p(group_):
    ro.globalenv["ft"] = ro.FloatVector(T); ro.globalenv["fs"] = ro.IntVector(S); ro.globalenv["gr"] = ro.StrVector(group_)
    ro.r("ci <- cuminc(ft, fs, gr, cencode = 0)")
    return float(ro.r("ci$Tests['1', 'pv']")[0]), float(ro.r("ci$Tests['2', 'pv']")[0])
OUTR["gray"] = {"class": gray_p(A.label.values), "gap": gray_p(np.where(A.gap365 == 1, "gap", "no gap"))}
print("Gray's test (events, deaths):", OUTR["gray"])

def cif730(d, cause=1):
    cd = AalenJohansenFitter(seed=42).fit(d.time, d.status, event_of_interest=cause).cumulative_density_
    return 100 * cd[cd.index <= 730].iloc[-1, 0]

groups = [("No gap >365 days", A[A.gap365 == 0], None), ("Gap >365 days", A[A.gap365 == 1], "gap365")] + \
         [(g, A[A.label == g], g) for g in ORDER]
t4 = []
for lab, d, key in groups:
    is_gap = lab.endswith("days")
    row = {"group": lab, "n": len(d), "events": int((d.status == 1).sum()), "deaths": int((d.status == 2).sum()),
           "incidence_730_pct": cif730(d)}
    for col, src in [("cs", "gap365" if is_gap else "class"), ("fg", "fg_gap365" if is_gap else "fg_class"),
                     ("dr", "dr_gap365" if is_gap else "dr_class")]:
        t = OUTR[src]["table"].get(key)
        row[f"{col}_HR"], row[f"{col}_lo"], row[f"{col}_hi"] = (t["HR"], t["lo"], t["hi"]) if t else (1.0, np.nan, np.nan)
    t4.append(row)
table4 = pd.DataFrame(t4)
print(table4.round(2).to_string(index=False))

In [ ]:
# Table S7: baseline characteristics by class (observed values)
B = A.copy()
grp_ = [("Overall", B)] + [(g, B[B.label == g]) for g in ORDER]
def smd_cont(x):
    s = [(B.loc[B.label == g, x].mean(), B.loc[B.label == g, x].std()) for g in ORDER]
    return max(abs(a[0] - b[0]) / np.sqrt((a[1] ** 2 + b[1] ** 2) / 2) for i, a in enumerate(s) for b in s[i + 1:])
def smd_bin(mask_):
    p = [mask_[B.label == g].mean() for g in ORDER]
    return max(abs(a - b) / np.sqrt((a * (1 - a) + b * (1 - b)) / 2) if 0 < a + b < 2 else 0
               for i, a in enumerate(p) for b in p[i + 1:])
mean_sd = lambda v: lambda d: f"{d[v].mean():.1f} ({d[v].std():.1f})"
med_iqr = lambda v, f=".0f": lambda d: f"{d[v].median():{f}} ({d[v].quantile(.25):{f}}-{d[v].quantile(.75):{f}})"
npct    = lambda fn: lambda d: f"{int(fn(d).sum()):,} ({100 * fn(d).mean():.1f})"

spec = [("Patients, n", lambda d: f"{len(d):,}", None),
        ("Age, years, mean (SD)", mean_sd("age_index"), smd_cont("age_index")),
        ("Female, n (%)", npct(lambda d: d.gender == "F"), smd_bin(B.gender == "F"))]
spec += [(f"Race: {r_}", npct(lambda d, r_=r_: d.race == r_), smd_bin(B.race == r_))
         for r_ in ["White", "Black", "Hispanic", "Asian", "Other/Unknown"]]
spec += [(f"Insurance: {i_}", npct(lambda d, i_=i_: d.ins_grp == i_), smd_bin(B.ins_grp == i_))
         for i_ in ["Medicare", "Medicaid", "Other"]]
spec += [("Systolic BP, mean (SD)", mean_sd("sbp_baseline"), smd_cont("sbp_baseline")),
         ("Diastolic BP, mean (SD)", mean_sd("dbp_baseline"), smd_cont("dbp_baseline")),
         ("eGFR, median (IQR)", med_iqr("egfr"), smd_cont("egfr")),
         ("HbA1c, median (IQR)", med_iqr("hba1c", ".1f"), smd_cont("hba1c")),
         ("BMI, mean (SD)", mean_sd("bmi"), smd_cont("bmi")),
         ("Elixhauser score, median (IQR)", med_iqr("elixhauser_vw"), smd_cont("elixhauser_vw")),
         ("Antihypertensive drugs, median (IQR)", med_iqr("n_antihtn_drugs"), smd_cont("n_antihtn_drugs")),
         ("Any emergency visit, n (%)", npct(lambda d: d.n_ed_prior > 0), smd_bin(B.n_ed_prior > 0)),
         ("Two or more inpatient stays, n (%)", npct(lambda d: d.n_inp_prior >= 2), smd_bin(B.n_inp_prior >= 2)),
         ("All encounters, median (IQR)", med_iqr("n_enc_prior"), smd_cont("n_enc_prior")),
         ("Admission beginning on index date, n (%)", npct(lambda d: d.has_idx_adm == 1), smd_bin(B.has_idx_adm == 1)),
         ("Missing eGFR, n (%)", npct(lambda d: d.egfr.isna()), smd_bin(B.egfr.isna())),
         ("Missing HbA1c, n (%)", npct(lambda d: d.hba1c.isna()), smd_bin(B.hba1c.isna())),
         ("Missing BMI, n (%)", npct(lambda d: d.bmi.isna()), smd_bin(B.bmi.isna())),
         ("Longest interval without contact, median (IQR)", med_iqr("gap_days"), smd_cont("gap_days")),
         ("Gap >365 days, n (%)", npct(lambda d: d.gap365 == 1), smd_bin(B.gap365 == 1)),
         ("CV hospitalization, n (%)", npct(lambda d: d.status == 1), smd_bin(B.status == 1)),
         ("Death without CV hospitalization, n (%)", npct(lambda d: d.status == 2), smd_bin(B.status == 2)),
         ("Follow-up, days, median (IQR)", med_iqr("time"), None)]
tableS7 = pd.DataFrame([{"characteristic": lab, **{g: fn(d) for g, d in grp_},
                         "max_smd": "" if s is None else f"{s:.2f}"} for lab, fn, s in spec])
pd.set_option("display.width", 250)
print(tableS7.to_string(index=False))

# Figure 3B: class makeup by gap status; Figure 3C-D: cumulative incidence curves by class
fig3b = pd.DataFrame([{"gap365": v, "label": g, "pct": 100 * (A[A.gap365 == v].label == g).mean()}
                      for v in [0, 1] for g in ORDER])
curves = []
for cause in [1, 2]:
    for g in ORDER:
        d = A[A.label == g]
        cd = AalenJohansenFitter(seed=42).fit(d.time, d.status, event_of_interest=cause).cumulative_density_
        cd = cd[cd.index <= 730]
        curves.append(pd.DataFrame({"cause": cause, "label": g, "day": cd.index.values.astype(float),
                                    "cif_pct": 100 * cd.iloc[:, 0].values.astype(float)}))
fig3cd = pd.concat(curves, ignore_index=True)

## 9 · Proportional hazards and minimum detectable hazard ratios

Scaled Schoenfeld residual tests (rank time transform) for the exposure terms, imputation 1. Minimum detectable hazard ratio with 80% power (Schoenfeld 1983) for each comparison.

In [ ]:
from lifelines.statistics import proportional_hazard_test
ph_p = {}
F1 = imputed(1)
for name, ex in [("gap365", G1), ("class", Cl)]:
    d = pd.concat([pd.DataFrame({"T": T, "E": E1}), build_X(F1, KN), ex], axis=1); d = d.loc[:, d.std() > 0]
    c = CoxPHFitter(penalizer=1e-6).fit(d, "T", "E")
    s = proportional_hazard_test(c, d, time_transform="rank").summary
    for k in ex.columns:
        if k in s.index: ph_p[k] = float(s.loc[k, "p"].iloc[0] if hasattr(s.loc[k, "p"], "iloc") else s.loc[k, "p"])
print("proportional hazards p by exposure term:", {k: round(v, 2) for k, v in ph_p.items()})
OUTR["ph_p"] = ph_p

mde = {"gap >365": schoenfeld_mde(int(E1.sum()), float(A.gap365.mean()))}
for g in ORDER[1:]:
    sub = A[A.label.isin(["Minimal Contact", g])]
    mde[g] = schoenfeld_mde(int((sub.status == 1).sum()), float((sub.label == g).mean()))
print("minimum detectable HR (80% power):", {k: tuple(round(x, 2) for x in v) for k, v in mde.items()})
OUTR["mde"] = mde

## 10 · Results summary

In [ ]:
f2 = lambda t: f"{t['HR']:.2f} ({t['lo']:.2f}-{t['hi']:.2f})"
print(f"LANDMARK POPULATION: n = {len(A):,}; cardiovascular hospitalizations {int(E1.sum()):,}; deaths without one {int(E2.sum()):,}")
print("\nTABLE 4A")
show = table4.copy()
for c in ["cs", "fg", "dr"]:
    show[c] = [f"{h:.2f} ({l:.2f}-{u:.2f})" if pd.notna(l) else "1 (reference)"
               for h, l, u in zip(show[f"{c}_HR"], show[f"{c}_lo"], show[f"{c}_hi"])]
print(show[["group", "n", "events", "deaths", "incidence_730_pct", "cs", "fg", "dr"]].round(1).to_string(index=False))
print(f"overall class p: cause-specific {OUTR['class']['p_overall']:.2f}, Fine-Gray {OUTR['fg_class']['p_overall']:.2f}, "
      f"doubly robust {OUTR['dr_class']['p_overall']:.2f}")
print(f"class and gap together: gap {f2(OUTR['class+gap365']['table']['gap365'])}, "
      f"{KEY} {f2(OUTR['class+gap365']['table'][KEY])}; "
      f"class given gap p {OUTR['class_given_gap_p']:.2f}; gap given class p {OUTR['gap_given_class_p']:.2f}")
print(f"gap >365 within Minimal Contact: {f2(OUTR['gap365_in_Minimal']['table']['gap365'])}")

print(f"\nGAP SPLINE (overall p {OUTR['gap_spline']['p_overall']:.2f}), HR vs a 90-day interval:")
print(curve[curve.gap_days.isin([180, 270, 366, 730])].round(2).to_string(index=False))
print(cuts[["HR", "lo", "hi", "prevalence_pct"]].round(2).to_string())

print("\nTABLE 4B: UNO'S C AT 730 DAYS")
print(uno.round(4).to_string(index=False))
print(f"pooled Wald p: gap {OUTR['gap365']['p_overall']:.2f}, class {OUTR['class']['p_overall']:.2f}")

print(f"\nGRAY'S TESTS (events, deaths): class {tuple(round(x, 3) for x in OUTR['gray']['class'])}, "
      f"gap {tuple(round(x, 3) for x in OUTR['gray']['gap'])}")
print(f"balance after weighting (largest SMD): {OUTR['balance_max_smd']}")
print(f"imputed vs observed means: { {c: tuple(round(x, 2) for x in v) for c, v in imp_means.items()} }; "
      f"largest fraction of missing information {OUTR['max_FMI']:.3f}")
print(f"proportional hazards, smallest exposure p: {min(ph_p.values()):.2f}")
print(f"minimum detectable HR (80% power): { {k: tuple(round(x, 2) for x in v) for k, v in mde.items()} }")

## 11 · Save

In [ ]:
save_df(pre, DATA_DIR, "covariates_preindex")
save_df(old_window, DATA_DIR, "covariates_oldwindow")
save_df(A, DATA_DIR, "analysis_frame")
save_df(MI, DATA_DIR, "mi")

for df_, name in [(table4, "table4"), (tableS7, "tableS7"), (curve, "figureS2_gap_spline"), (cuts.reset_index(), "gap_cutoffs"),
                  (uno, "uno_c"), (fig3b, "figure3b"), (fig3cd, "figure3cd")]:
    df_.to_csv(os.path.join(RESULTS_DIR, f"{name}.csv"), index=False)
json.dump(KN, open(os.path.join(RESULTS_DIR, "spline_knots.json"), "w"), indent=1)
OUTR.update({"n": int(len(A)), "events": int(E1.sum()), "deaths": int(E2.sum()), "imputed_vs_observed_means": imp_means})
save_json(OUTR, RESULTS_DIR, "06_main")